# Stream a Response via our Cortex Platform

This example shows how to stream a response from the Snowflake Cortex platform using our platform client.

## Setup

In [ ]:
from typing import cast

from utils import MockKernel, setup_notebook

from agent_platform.core.kernel import Kernel

if not setup_notebook():
    raise ValueError("Failed to setup notebook")

In [ ]:
from agent_platform.core.platforms.cortex import CortexPlatformParameters
from agent_platform.core.platforms.cortex.configs import CortexModelMap

# Configurations that will be used
default_model_map = CortexModelMap.get_default_instance()

# Platform Parameters
parameters = CortexPlatformParameters()

## Create a Platform Client

In [ ]:
from agent_platform.core.platforms.cortex.client import CortexClient

cortex_client = CortexClient(
    parameters=parameters,
)

cortex_client.attach_kernel(cast(Kernel, MockKernel()))

## Create a Tool

In [ ]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False


joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)

## Create a Prompt

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[
        PromptTextContent(
            text="Please use the joke_judge tool to "
            "judge if the following joke is funny: "
            '"Why did the chicken cross the road?"\n'
            '"To get to the other side!"',
        ),
    ],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool],
)

await general_prompt.finalize_messages()
cortex_prompt = await cortex_client.converters.convert_prompt(general_prompt)

print(cortex_prompt)

## Request and Stream a Response

In [ ]:
deltas = []
i = 0
async for delta in cortex_client.generate_stream_response(
    cortex_prompt,
    "claude-3-5-sonnet",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [ ]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembeld_dict = combine_generic_deltas(deltas)
pprint(assembeld_dict)

### Now as a ResponseMessage

In [ ]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembeld_dict)
pprint(response_message)

# Continue the Conversation

## Execute the Tool

In [ ]:
from agent_platform.core.responses import ResponseToolUseContent

tool_use_content_block = response_message.content[1]
assert isinstance(tool_use_content_block, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content block of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [ ]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block, ResponseToolUseContent)
tool_result_content = PromptToolResultContent(
    tool_name=tool_use_content_block.tool_name,
    tool_call_id=tool_use_content_block.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

print(tool_result_content)

## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [ ]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use = response_message.content[1]
assert isinstance(original_tool_use, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use = ThreadToolUsageContent.from_response_tool_use(original_tool_use)
prompt_tool_use = PromptToolUseContent.from_thread_tool_usage(ai_tool_use)

pprint(prompt_tool_use)

## Create new Prompt

In [ ]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool],
)
await return_gen_prompt.finalize_messages()
return_cortex_prompt = await cortex_client.converters.convert_prompt(
    return_gen_prompt,
)

pprint(return_cortex_prompt)

## Generate a new Response (Non-Streaming)

In [ ]:
response = await cortex_client.generate_response(
    return_cortex_prompt,
    "claude-3-5-sonnet",
)

pprint(response)

In [ ]:
# NOTE: something fishy is happening here, tool_calls are getting dropped non-streaming
# Reporting this to Snowflake to see what's up (perhaps my request structured is wrong)
response = await cortex_client.generate_response(
    cortex_prompt,
    "claude-3-5-sonnet",
)

pprint(response)